# Laboratorio 02.1 — Fundamentos de prompts

Separaremos instrucciones, tarea y entrada para construir prompts que puedan inspeccionarse, versionarse y evaluarse antes de invocar un modelo.

## 1. Objetivos de aprendizaje

Al finalizar podrás:
- Distinguir la instrucción del sistema de la entrada del usuario.
- Construir prompts a partir de campos explícitos en vez de concatenaciones ambiguas.
- Comparar versiones de una instrucción con la entrada fija.
- Medir una línea base local y mantener opcional la única llamada remota.

## 2. Conceptos

Un prompt es parte de la interfaz de un sistema: define tarea, contexto, restricciones y forma de respuesta. Las instrucciones deben ser específicas, observables y compatibles con la evaluación. Una mejora aparente en un ejemplo no demuestra generalización; conviene fijar entradas, registrar la versión del prompt y medir un conjunto representativo.

## 3. Arquitectura

```mermaid
flowchart LR
    A[Instruccion del sistema] --> C[Constructor de prompt]
    B[Entrada del usuario] --> C
    C --> D[Prompt versionado]
    D --> E[Modelo opcional]
    B --> F[Linea base local]
    F --> G[Evaluacion]
    E --> G
```

## 4. Preparación del entorno

Cargamos las dependencias locales y la raíz del curso. El experimento predeterminado no crea un cliente ni realiza una llamada.

In [1]:
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path
import sys

raices_candidatas = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
raiz_curso = next(
    (
        candidato
        for raiz in raices_candidatas
        for candidato in (raiz / 'genai', raiz)
        if (candidato / 'src' / 'genai').is_dir()
    ),
    None,
)
if raiz_curso is None:
    raise FileNotFoundError('No se encontró el proyecto genAI con src/genai.')

sys.path.insert(0, str(raiz_curso / 'src'))

In [2]:
import re
from time import perf_counter

import pandas as pd

from genai.environment import preparar_entorno
from genai.models import get_llm

raiz_curso = preparar_entorno()

## 5. Definición del problema

Clasificaremos consultas de soporte en una categoría. El dataset es pequeño y escrito para el laboratorio; se utiliza para ilustrar exact match y accuracy, no para afirmar rendimiento en producción. La misma entrada de ejemplo servirá al comparar redacciones de instrucciones.

## 6. Línea base

Una regla local detecta algunas palabras clave sin un LLM. Esta línea base establece un punto de comparación barato y determinista; sus fallos son útiles para identificar qué información falta en la instrucción.

In [3]:
datos = [
    {'texto': 'No he recibido el paquete que pedí', 'categoria': 'envios'},
    {'texto': '¿Dónde puedo seguir mi pedido?', 'categoria': 'envios'},
    {'texto': 'Quiero devolver este producto', 'categoria': 'devoluciones'},
    {'texto': 'Necesito un reembolso por mi compra', 'categoria': 'devoluciones'},
    {'texto': 'Veo un cobro duplicado en mi tarjeta', 'categoria': 'pagos'},
    {'texto': 'Quiero cambiar la talla de mi pedido', 'categoria': 'cambios'},
]

def clasificar_por_reglas(texto: str) -> str:
    """Clasifica consultas con reglas locales simples."""
    texto_normalizado = texto.casefold()
    if any(palabra in texto_normalizado for palabra in ('devolver', 'devolución', 'reembolso')):
        return 'devoluciones'
    if any(palabra in texto_normalizado for palabra in ('cobro', 'tarjeta', 'pago')):
        return 'pagos'
    if any(palabra in texto_normalizado for palabra in ('cambiar', 'talla')):
        return 'cambios'
    if any(palabra in texto_normalizado for palabra in ('paquete', 'pedido', 'recibido', 'seguir')):
        return 'envios'
    return 'sin_clasificar'

base = pd.DataFrame(datos)
base['prediccion_reglas'] = base['texto'].map(clasificar_por_reglas)
base['exact_match'] = base['prediccion_reglas'] == base['categoria']
display(base)
print(f'Accuracy de la linea base: {base["exact_match"].mean():.3f}')

,texto,categoria,prediccion_reglas,exact_match
0,No he recibido el paquete que pedí,envios,envios,True
1,¿Dónde puedo seguir mi pedido?,envios,envios,True
2,Quiero devolver este producto,devoluciones,devoluciones,True
3,Necesito un reembolso por mi compra,devoluciones,devoluciones,True
4,Veo un cobro duplicado en mi tarjeta,pagos,pagos,True
5,Quiero cambiar la talla de mi pedido,cambios,cambios,True


Accuracy de la linea base: 1.000


## 7. Implementación

El constructor mantiene por separado instrucción y consulta. Una versión del prompt se registra junto al texto para que cambios posteriores sean atribuibles y revisables.

In [4]:
VERSION_PROMPT = 'clasificacion-soporte-v1'

def construir_prompt(instruccion: str, consulta: str) -> str:
    """Construye un prompt de clasificacion con secciones explicitas."""
    return (
        f'INSTRUCCION\n{instruccion.strip()}\n\n'
        f'CONSULTA\n{consulta.strip()}\n\n'
        'Responde solo con una categoria: envios, devoluciones, pagos o cambios.'
    )

instruccion_base = 'Clasifica la consulta de soporte en la categoria que mejor corresponda.'
consulta_ejemplo = '¿Cómo puedo devolver mi compra?'
prompt_base = construir_prompt(instruccion_base, consulta_ejemplo)
print(prompt_base)

INSTRUCCION
Clasifica la consulta de soporte en la categoria que mejor corresponda.

CONSULTA
¿Cómo puedo devolver mi compra?

Responde solo con una categoria: envios, devoluciones, pagos o cambios.


## 8. Experimentos

Comparamos dos instrucciones sobre la misma consulta y mantenemos idéntica la restricción de salida. La longitud aproximada se cuenta en palabras, no en tokens del modelo; el objetivo es observar el costo de instrucciones más explícitas.

In [5]:
instrucciones = {
    'breve': 'Clasifica la consulta de soporte.',
    'explicita': 'Identifica el motivo principal de la consulta. Elige la categoria que mejor coincida con la solicitud completa.',
}
experimentos_prompt = pd.DataFrame([
    {
        'version': nombre,
        'palabras_aprox': len(construir_prompt(instruccion, consulta_ejemplo).split()),
        'prompt': construir_prompt(instruccion, consulta_ejemplo),
    }
    for nombre, instruccion in instrucciones.items()
])
display(experimentos_prompt[['version', 'palabras_aprox']])
experimentos_prompt.to_csv(raiz_curso / 'outputs' / '02_01_prompt_versions.csv', index=False)

,version,palabras_aprox
0,breve,22
1,explicita,34


## 9. Evaluación

Exact match compara la etiqueta predicha con la referencia; accuracy es la proporción total correcta. Antes de evaluar un prompt con un modelo, fija el dataset, la versión, el proveedor y el número de llamadas. Aquí evaluamos la línea base local para mantener cero costo.

In [6]:
metricas_base = pd.DataFrame([{
    'prompt_version': VERSION_PROMPT,
    'exact_match': float(base['exact_match'].mean()),
    'accuracy': float((base['prediccion_reglas'] == base['categoria']).mean()),
    'llamadas_llm': 0,
}])
display(metricas_base)
metricas_base.to_csv(raiz_curso / 'outputs' / '02_01_baseline_metrics.csv', index=False)

EJECUTAR_LLAMADA = False
PROVEEDOR = 'ollama'
if EJECUTAR_LLAMADA:
    inicio = perf_counter()
    modelo = get_llm(PROVEEDOR)
    respuesta = modelo.invoke(prompt_base)
    print(f'Latencia: {perf_counter() - inicio:.2f} s')
    print(respuesta.content)

,prompt_version,exact_match,accuracy,llamadas_llm
0,clasificacion-soporte-v1,1.0,1.0,0


## 10. Discusión

Una instrucción más larga puede reducir ambigüedad, pero también consume contexto y no garantiza mejor clasificación. La línea base puede ser sorprendentemente fuerte en datos sencillos; las mejoras del modelo deben medirse sobre consultas variadas. El bloque opcional realiza como máximo una llamada y debe activarse deliberadamente.

## 11. Ejercicios

1. Añade consultas ambiguas y etiqueta cuáles requieren pedir aclaración.
2. Escribe dos versiones de instrucción con una sola diferencia y documenta su hipótesis.
3. Calcula accuracy por categoría además de accuracy global.
4. Ejecuta una llamada local con Ollama y anota el modelo configurado.

## 12. Desafío

Diseña una evaluación de regresión para comparar tres versiones del prompt sobre un conjunto de consultas no usado al redactarlas. Registra versión, modelo, latencia, llamadas, etiquetas y fallos de análisis.

## 13. Ideas clave

- Un prompt es una interfaz que debe poder versionarse.
- Separa instrucciones de datos de entrada.
- Conserva una línea base simple para medir mejoras reales.
- La llamada al modelo es opcional, explícita y limitada a una solicitud.